## Coffee Shop Sales: Data Preparation

### 1. Configure and start Spark

In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])

JAVA_HOME: C:\Program Files\Java\jdk-17


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder
    .appName("CoffeeShopProject")
    .master("local[*]")
    .getOrCreate()
)
print("Spark version:", spark.version)
print("Spark session:", spark.sparkContext.appName)

Spark version: 4.2.0
Spark session: CoffeeShopProject


### 2. Load and inspect the source data

In [4]:
file_path = r"D:\Projects\Coffee Shop Sales - Cluster Computing\Dataset\Coffee Shop Sales.csv"
df = spark.read.csv(
    file_path,
    header = True,
    inferSchema = True
)

In [5]:
df.show(5)

+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+--------------------+--------------------+
|transaction_id|transaction_date|   transaction_time|transaction_qty|store_id| store_location|product_id|unit_price|  product_category|        product_type|      product_detail|
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+--------------------+--------------------+
|             1|      01-01-2023|2026-10-24 07:06:11|              2|       5|Lower Manhattan|        32|       3.0|            Coffee|Gourmet brewed co...|         Ethiopia Rg|
|             2|      01-01-2023|2026-10-24 07:08:56|              2|       5|Lower Manhattan|        57|       3.1|               Tea|     Brewed Chai tea|Spicy Eye Opener ...|
|             3|      01-01-2023|2026-10-24 07:14:04|              2|       5|Lower Manhattan|        59|     

In [6]:
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()

Total rows: 149116
Total columns: 11
root
 |-- transaction_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- transaction_qty: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_location: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- product_detail: string (nullable = true)



In [7]:
df.show(10, truncate=False)

+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|transaction_id|transaction_date|transaction_time   |transaction_qty|store_id|store_location |product_id|unit_price|product_category  |product_type         |product_detail             |
+--------------+----------------+-------------------+---------------+--------+---------------+----------+----------+------------------+---------------------+---------------------------+
|1             |01-01-2023      |2026-10-24 07:06:11|2              |5       |Lower Manhattan|32        |3.0       |Coffee            |Gourmet brewed coffee|Ethiopia Rg                |
|2             |01-01-2023      |2026-10-24 07:08:56|2              |5       |Lower Manhattan|57        |3.1       |Tea               |Brewed Chai tea      |Spicy Eye Opener Chai Lg   |
|3             |01-01-2023      |2026-10-24 07:14:04|2              |5

In [8]:
from pyspark.sql.functions import col, sum
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+
|transaction_id|transaction_date|transaction_time|transaction_qty|store_id|store_location|product_id|unit_price|product_category|product_type|product_detail|
+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+
|             0|               0|               0|              0|       0|             0|         0|         0|               0|           0|             0|
+--------------+----------------+----------------+---------------+--------+--------------+----------+----------+----------------+------------+--------------+



### 3. Check data completeness

In [9]:
df.describe().show()

+-------+-----------------+----------------+------------------+------------------+---------------+-----------------+-----------------+----------------+----------------+--------------------+
|summary|   transaction_id|transaction_date|   transaction_qty|          store_id| store_location|       product_id|       unit_price|product_category|    product_type|      product_detail|
+-------+-----------------+----------------+------------------+------------------+---------------+-----------------+-----------------+----------------+----------------+--------------------+
|  count|           149116|          149116|            149116|            149116|         149116|           149116|           149116|          149116|          149116|              149116|
|   mean|74737.37187156308|            NULL| 1.438276241315486| 5.342062555326055|           NULL|47.91860699053086|3.382219413074453|            NULL|            NULL|                NULL|
| stddev|43153.60001591786|            NULL|0.5425